# Phase 5 - End-to-End Evaluation

This notebook checks the whole RadScribe decision flow on the full test split.

The main Phase 5 number should not come from a few nice examples. It should come from every test study.

## Plan

1. Run the vision model on all test images.
2. Pool predictions at study level.
3. Apply the same Phase 4 gate: main finding at `0.70`, borderline at `0.50`.
4. Compute full-test decision metrics without using the LLM.
5. Run the full agent only on the studies where the gate fires.
6. Keep the small example cases and critic stress test as a showcase, not as the main metric.

In [7]:
from pathlib import Path
import ast
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "manifest.parquet"
OUT_DIR = PROJECT_ROOT / "outputs" / "agent_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GATE_RESULTS_PATH = OUT_DIR / "full_test_gate_results.csv"
GATE_SUMMARY_PATH = OUT_DIR / "full_test_gate_summary.json"
PER_FINDING_PATH = OUT_DIR / "full_test_per_finding_metrics.csv"
AGENT_RESULTS_PATH = OUT_DIR / "agent_firing_subset_results.csv"
SHOWCASE_PATH = OUT_DIR / "showcase_cases.csv"
CRITIC_STRESS_PATH = OUT_DIR / "critic_stress_test.json"

DISEASE_LABELS = [
    "Cardiomegaly",
    "Atelectasis",
    "Consolidation / Pneumonia",
    "Pleural Effusion",
    "Edema",
    "Pneumothorax",
]

SUPPORTED_LABELS = [
    "Cardiomegaly",
    "Atelectasis",
    "Consolidation / Pneumonia",
    "Pleural Effusion",
]

MAIN_THRESHOLD = 0.70
BORDERLINE_THRESHOLD = 0.50
RANDOM_STATE = 5

print(PROJECT_ROOT)

e:\Project\RadScribe


## Load Test Studies

In [8]:
def labels_as_list(value):
    if isinstance(value, list):
        return value
    if hasattr(value, "tolist"):
        return list(value.tolist())
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (SyntaxError, ValueError):
            pass
        return [part.strip() for part in value.split("|") if part.strip()]
    return []


def has_any_disease(labels):
    labels = set(labels)
    return any(label in labels for label in DISEASE_LABELS)


df = pd.read_parquet(MANIFEST_PATH)
test_images = df[df["split"].eq("test")].copy()

study_truth = test_images.drop_duplicates("study_id").copy()
study_truth["true_labels_list"] = study_truth["labels"].apply(labels_as_list)
study_truth["true_has_disease"] = study_truth["true_labels_list"].apply(has_any_disease)
study_truth["true_is_normal_or_other"] = ~study_truth["true_has_disease"]

print("test images:", len(test_images))
print("test studies:", study_truth["study_id"].nunique())
study_truth["true_has_disease"].value_counts(dropna=False)

test images: 564
test studies: 550


true_has_disease
False    443
True     107
Name: count, dtype: int64

## Full Test Vision + Gate Pass

This part does not call the LLM. It only uses the vision model and the same gate used by the agent.

In [9]:
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from src.models.predict import load_finetuned_model
from src.models.dataset import eval_transform


class XrayRows(Dataset):
    def __init__(self, rows):
        self.rows = rows.reset_index(drop=True)
        self.transform = eval_transform()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        image_path = PROJECT_ROOT / row["image_path"]
        image = Image.open(image_path).convert("RGB")
        return {
            "study_id": row["study_id"],
            "image_path": row["image_path"],
            "image": self.transform(image),
        }


def collate_batch(batch):
    return {
        "study_id": [row["study_id"] for row in batch],
        "image_path": [row["image_path"] for row in batch],
        "image": torch.stack([row["image"] for row in batch]),
    }


model, device = load_finetuned_model()
loader = DataLoader(
    XrayRows(test_images),
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_batch,
)

image_rows = []
with torch.no_grad():
    for batch in loader:
        images = batch["image"].to(device)
        probs = torch.sigmoid(model(images)).cpu().numpy()

        for i, study_id in enumerate(batch["study_id"]):
            row = {
                "study_id": study_id,
                "image_path": batch["image_path"][i],
            }
            for j, label in enumerate(DISEASE_LABELS):
                row[label] = float(probs[i, j])
            image_rows.append(row)

image_pred = pd.DataFrame(image_rows)
study_pred = image_pred.groupby("study_id", as_index=False)[DISEASE_LABELS].mean()
study_pred.head()

,study_id,Cardiomegaly,Atelectasis,Consolidation / Pneumonia,Pleural Effusion,Edema,Pneumothorax
0,1,0.360607,0.339335,0.334630,0.108075,0.180424,0.249075
1,1009,0.068519,0.060606,0.058575,0.021542,0.015194,0.053129
2,1010,0.296614,0.113995,0.166984,0.044279,0.036434,0.052804
3,1018,0.067748,0.130850,0.131893,0.037810,0.015441,0.038991
4,1020,0.563539,0.662537,0.390281,0.231393,0.143978,0.330083


## Study-Level Decision Metrics

This is the real Phase 5 denominator. It uses all test studies.

In [10]:
def gate_row(row):
    main = [label for label in DISEASE_LABELS if row[label] >= MAIN_THRESHOLD]
    borderline = [
        label
        for label in DISEASE_LABELS
        if BORDERLINE_THRESHOLD <= row[label] < MAIN_THRESHOLD
    ]

    if main:
        status = "high_confidence"
    elif borderline:
        status = "borderline"
    else:
        status = "no_finding_above_threshold"

    return pd.Series(
        {
            "vision_status": status,
            "main_findings": json.dumps(main),
            "borderline_findings": json.dumps(borderline),
            "would_draft": bool(main),
        }
    )


gate = study_pred.merge(
    study_truth[["study_id", "image_path", "true_labels_list", "true_has_disease", "true_is_normal_or_other"]],
    on="study_id",
    how="left",
)

gate = pd.concat([gate, gate.apply(gate_row, axis=1)], axis=1)
gate["true_labels"] = gate["true_labels_list"].apply(lambda labels: json.dumps(labels))
gate["true_label_in_main"] = gate.apply(
    lambda row: bool(set(row["true_labels_list"]).intersection(json.loads(row["main_findings"]))),
    axis=1,
)

tp = int((gate["true_has_disease"] & gate["would_draft"]).sum())
fn = int((gate["true_has_disease"] & ~gate["would_draft"]).sum())
fp = int((~gate["true_has_disease"] & gate["would_draft"]).sum())
tn = int((~gate["true_has_disease"] & ~gate["would_draft"]).sum())

summary = {
    "n_test_studies": int(len(gate)),
    "true_disease_studies": int(gate["true_has_disease"].sum()),
    "normal_or_other_studies": int((~gate["true_has_disease"]).sum()),
    "would_draft_count": int(gate["would_draft"].sum()),
    "tp": tp,
    "fp": fp,
    "tn": tn,
    "fn": fn,
    "sensitivity": tp / (tp + fn) if (tp + fn) else None,
    "specificity": tn / (tn + fp) if (tn + fp) else None,
    "false_report_rate": fp / (fp + tn) if (fp + tn) else None,
    "true_label_in_main_rate_when_drafted": float(gate.loc[gate["would_draft"], "true_label_in_main"].mean()) if gate["would_draft"].any() else None,
}

gate.to_csv(GATE_RESULTS_PATH, index=False)
GATE_SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

pd.DataFrame([summary])

,n_test_studies,true_disease_studies,normal_or_other_studies,would_draft_count,tp,fp,tn,fn,sensitivity,specificity,false_report_rate,true_label_in_main_rate_when_drafted
0,550,107,443,124,66,58,385,41,0.616822,0.869074,0.130926,0.451613


## Per-Finding Precision And Recall

I report the four supported findings here. Edema and Pneumothorax are too rare in this dataset, so I keep them out of the headline metrics.

In [11]:
metric_rows = []
for label in SUPPORTED_LABELS:
    y_true = gate["true_labels_list"].apply(lambda labels: label in labels)
    y_pred = gate["main_findings"].apply(lambda text: label in json.loads(text))

    tp_l = int((y_true & y_pred).sum())
    fp_l = int((~y_true & y_pred).sum())
    fn_l = int((y_true & ~y_pred).sum())

    metric_rows.append(
        {
            "label": label,
            "positives": int(y_true.sum()),
            "predicted_positive": int(y_pred.sum()),
            "tp": tp_l,
            "fp": fp_l,
            "fn": fn_l,
            "precision": tp_l / (tp_l + fp_l) if (tp_l + fp_l) else None,
            "recall": tp_l / (tp_l + fn_l) if (tp_l + fn_l) else None,
        }
    )

per_finding = pd.DataFrame(metric_rows)
per_finding.to_csv(PER_FINDING_PATH, index=False)
per_finding

,label,positives,predicted_positive,tp,fp,fn,precision,recall
0,Cardiomegaly,44,93,29,64,15,0.311828,0.659091
1,Atelectasis,44,53,21,32,23,0.396226,0.477273
2,Consolidation / Pneumonia,28,46,9,37,19,0.195652,0.321429
3,Pleural Effusion,22,30,13,17,9,0.433333,0.590909


## Miss Analysis

These are the cases I would read by hand first.

In [12]:
false_reports = gate[(~gate["true_has_disease"]) & gate["would_draft"]].copy()
missed_disease = gate[gate["true_has_disease"] & ~gate["would_draft"]].copy()
wrong_main = gate[gate["would_draft"] & ~gate["true_label_in_main"]].copy()

print("false reports:", len(false_reports))
print("missed disease studies:", len(missed_disease))
print("drafted but no true label in main findings:", len(wrong_main))

false_reports[["study_id", "true_labels", "main_findings", "borderline_findings"]].head(10)

false reports: 58
missed disease studies: 41
drafted but no true label in main findings: 68


,study_id,true_labels,main_findings,borderline_findings
10,106,"[""No Finding""]","[""Cardiomegaly""]",[]
15,1081,"[""Other""]","[""Cardiomegaly"", ""Atelectasis"", ""Consolidation...",[]
42,1221,"[""Other""]","[""Cardiomegaly""]",[]
60,1311,"[""Other""]","[""Cardiomegaly""]",[]
119,1697,"[""No Finding""]","[""Atelectasis""]","[""Consolidation / Pneumonia""]"
122,172,"[""Other""]","[""Cardiomegaly""]",[]
124,1735,"[""Other""]","[""Atelectasis""]","[""Consolidation / Pneumonia"", ""Pleural Effusio..."
133,1789,"[""Other""]","[""Atelectasis""]","[""Cardiomegaly"", ""Consolidation / Pneumonia""]"
139,1827,"[""No Finding""]","[""Cardiomegaly""]",[]
158,1934,"[""Other""]","[""Atelectasis"", ""Consolidation / Pneumonia"", ""...","[""Cardiomegaly"", ""Pneumothorax""]"


## Full Agent Run On The Firing Subset

This is the LLM part. Only studies that passed the main vision gate are run through retrieval, draft, critic, disclaimer, and trace logging.

In [13]:
from src.agent.nodes import DISCLAIMER
from src.agent.run import run_agent


RUN_AGENT_ON_FIRING_SUBSET = True
MAX_AGENT_RUNS = None  # use a small number like 20 while testing; None means all firing studies


def run_one_agent_case(row):
    result = run_agent(PROJECT_ROOT / row["image_path"])
    final_report = result.get("final_report", "")

    return {
        "study_id": row["study_id"],
        "image_path": row["image_path"],
        "true_labels": row["true_labels"],
        "main_findings": json.dumps(result.get("main_findings", [])),
        "borderline_findings": json.dumps(result.get("borderline_findings", [])),
        "best_retrieval_score": result.get("best_retrieval_score"),
        "retrieval_ok": result.get("retrieval_ok"),
        "can_draft": result.get("can_draft", False),
        "critic_supported": result.get("critic_result", {}).get("supported"),
        "has_disclaimer": DISCLAIMER in final_report,
        "trace_path": result.get("trace_path"),
        "final_report": final_report,
    }


if RUN_AGENT_ON_FIRING_SUBSET:
    firing = gate[gate["would_draft"]].copy()
    if MAX_AGENT_RUNS is not None:
        firing = firing.head(MAX_AGENT_RUNS)

    agent_rows = [run_one_agent_case(row) for _, row in firing.iterrows()]
    agent_results = pd.DataFrame(agent_rows)
    agent_results.to_csv(AGENT_RESULTS_PATH, index=False)
else:
    agent_results = pd.DataFrame()

agent_results[["study_id", "true_labels", "main_findings", "critic_supported", "has_disclaimer"]].head()

,study_id,true_labels,main_findings,critic_supported,has_disclaimer
0,1021,"[""Atelectasis"", ""Pneumothorax""]","[""Atelectasis"", ""Cardiomegaly"", ""Consolidation...",True,True
1,106,"[""No Finding""]","[""Cardiomegaly""]",True,True
2,1081,"[""Other""]","[""Cardiomegaly"", ""Pleural Effusion"", ""Atelecta...",True,True
3,1169,"[""Atelectasis""]","[""Pleural Effusion"", ""Consolidation / Pneumoni...",True,True
4,1170,"[""Cardiomegaly""]","[""Cardiomegaly""]",True,True


## Agent Factuality Summary

This uses the critic result on the firing subset. It checks whether the drafted text stayed inside the retrieved evidence.

In [14]:
if AGENT_RESULTS_PATH.exists():
    agent_results = pd.read_csv(AGENT_RESULTS_PATH)

if len(agent_results):
    factuality = {
        "n_agent_runs": int(len(agent_results)),
        "critic_supported_rate": float(agent_results["critic_supported"].mean()),
        "disclaimer_rate": float(agent_results["has_disclaimer"].mean()),
        "retrieval_ok_rate": float(agent_results["retrieval_ok"].mean()),
        "mean_best_retrieval_score": float(agent_results["best_retrieval_score"].mean()),
    }
else:
    factuality = {"n_agent_runs": 0}

factuality

{'n_agent_runs': 124,
 'critic_supported_rate': 0.9516129032258065,
 'disclaimer_rate': 1.0,
 'retrieval_ok_rate': 1.0,
 'mean_best_retrieval_score': 0.7576571438581713}

## Small Showcase Cases

These examples are still useful, but they are not the main metric. They help show the main paths in a readable way.

In [15]:
normal_case = gate[(~gate["true_has_disease"]) & ~gate["would_draft"]].sample(1, random_state=RANDOM_STATE)
positive_case = gate[gate["true_has_disease"] & gate["would_draft"]].sample(1, random_state=RANDOM_STATE)
false_report_case = false_reports.sample(1, random_state=RANDOM_STATE) if len(false_reports) else pd.DataFrame()

showcase = pd.concat([positive_case, normal_case, false_report_case], ignore_index=True)
showcase["case_note"] = ["positive draft", "quiet normal/other", "false report"][: len(showcase)]
showcase[["case_note", "study_id", "true_labels", "main_findings", "borderline_findings"]].to_csv(SHOWCASE_PATH, index=False)
showcase[["case_note", "study_id", "true_labels", "main_findings", "borderline_findings"]]

,case_note,study_id,true_labels,main_findings,borderline_findings
0,positive draft,2901,"[""Atelectasis""]","[""Atelectasis""]","[""Cardiomegaly"", ""Consolidation / Pneumonia"", ..."
1,quiet normal/other,1413,"[""Other""]",[],"[""Cardiomegaly"", ""Atelectasis""]"
2,false report,2931,"[""Other""]","[""Atelectasis""]","[""Consolidation / Pneumonia""]"


## Critic Stress Test

This plants a claim that should not be supported by the evidence. The critic should return `supported = false`.

In [16]:
from src.agent.nodes import critic_node

assert len(agent_results) > 0, "Run the firing subset agent pass before the critic stress test."

trace_path = Path(agent_results.iloc[0]["trace_path"])
trace = json.loads(trace_path.read_text(encoding="utf-8"))

bad_state = {
    **trace,
    "draft_report": (
        trace.get("draft_report", "")
        + "\n\nAdditional unsupported claim: a large pneumothorax is present."
    ),
}

critic_test = critic_node(bad_state)
critic_result = critic_test.get("critic_result", {})
CRITIC_STRESS_PATH.write_text(json.dumps(critic_result, indent=2), encoding="utf-8")
critic_result

{'supported': False,
 'missing_evidence': ['presence of pleural effusion',
  'a large pneumothorax is present'],
 'safety_note': 'The draft includes unsupported claims regarding pleural effusion and pneumothorax.'}

## Takeaway

The full test split has `550` studies. The agent would draft for `124` studies and stay quiet for `426` studies.

- Sensitivity: `0.617`.
- Specificity: `0.869`.
- False-report rate on normal/Other studies: `0.131`.
- True-label-in-main rate when drafted: `0.452`.
- Firing subset agent runs: `124`.
- Retrieval OK rate on drafted cases: `1.000`.
- Critic supported rate: `0.952`.
- Disclaimer rate: `1.000`.
- Critic stress test: supported = `false`; it flagged unsupported pleural effusion and large pneumothorax claims.

The main result is mixed, and that is important. The safety wrapper behaves well: the disclaimer is always present, retrieval works for drafted cases, and the critic can catch a planted unsupported claim. But the full-test numbers also show the weak point clearly. The agent still makes false disease drafts when the vision model is confidently wrong. So the system is useful as a safety-aware prototype, but not reliable enough for clinical use.